In [1]:
%cd drive/MyDrive/Thesis/Models/PLM_embedding

/content/drive/MyDrive/Thesis/Models/PLM_embedding


In [2]:
!ls

get_embeddings.ipynb  model.py	   saved_model
inference.ipynb       __pycache__  training.ipynb


In [3]:
%cp -R '../../Data/training_embeddings' '/home/training_embeddings'

In [4]:
import numpy as np
import torch
import torch.nn as nn
import gc

from model import CNN_BRNN

In [5]:
class RAM_dataset(torch.utils.data.Dataset):
    def __init__(self, tensor_dict, np_data):
        self.tensor_dict = tensor_dict
        self.samples = list(tensor_dict.keys())
        self.np_data = np_data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # PSSM  = self.np_data[self.samples[idx]][3:]
        embeddings = self.tensor_dict[self.samples[idx]]
        mask  = self.np_data[self.samples[idx]][1]
        label = self.np_data[self.samples[idx]][2]       
        label[mask==0] = -9
        label = torch.from_numpy(label).long()

        embeddings = embeddings.transpose(0,1)
        if embeddings.shape[-1] >= 700:
          embeddings = embeddings[:,:700]
          label      = torch.nn.functional.pad(label, (1,1), 'constant', -9)[:700]
        else:
          embeddings = torch.nn.functional.pad(embeddings, (700 - embeddings.shape[1], 0), 'constant',  0)
          label      = torch.nn.functional.pad(label, (699 - label.shape[0], 1), 'constant', -9)
        return embeddings, label


In [12]:
#----- training loop
def training_loop(epochs, np_data, model, loss_func, opt):
  for epoch in range(epochs):
    print(f"Epoch[{epoch+1}/{epochs}]\n")

    acc_num = 0
    acc_den = 0

    #load half the training data to be used for each epoch into RAM  # randomly pick 1/2 the files to use for the epoch. Retain the other 1/2 for next epoch
    #readign from disc takes too long, and dataset is too large to load into RAM all at once.
    if epoch%2 == 0 :      
      rand_index = np.arange(21005)
      np.random.shuffle(rand_index)
      include_chains = rand_index[:10503]
    else:
      include_chains = rand_index[10503:]

    print("loading data {}".format(len(include_chains)))
    RAM_tensor_dict = {np_data.files[x] : torch.load('/home/training_embeddings/'+np_data.files[x]+'.pt') for x in include_chains}
    print("data loaded")
    my_dataset = RAM_dataset(RAM_tensor_dict, np_data)
    batch_size=64
    dataloader = torch.utils.data.DataLoader(my_dataset,shuffle=True,batch_size=batch_size) #torch.save zips the files, and pytorch zip module has some problem when using multiple workers (when two batches try to open the same filename)

    for x,y in dataloader:      
      #forward
      x= x.to(device='cuda')
      y= y.to(device='cuda')
      outputs    = model(x)

      # print(outputs.shape)
      # output : (N x C x L)
      _, preds   = torch.max(outputs, 1)

      acc_num += (preds == y).sum().item() 
      acc_den += len(y[y > -1])

      loss = loss_func(outputs, y)

      #backward
      opt.zero_grad() 
      loss.backward()

      #adam step
      opt.step()

      #scheduler step

    print(f'Accuracy: {(acc_num/acc_den):.3f}\n')
    del RAM_tensor_dict
    del my_dataset
    del dataloader
    gc.collect()
    # for x,y in validation_loader:


In [8]:
device = "cuda"

np_data = np.load('../../Data/PSSM_train_data.npz')

model = CNN_BRNN().to(device)
model.load_state_dict(torch.load('./saved_model'))
model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay = 0.01)
loss_func = torch.nn.CrossEntropyLoss(ignore_index = -9)

In [ ]:
training_loop(epochs=10, np_data=np_data, model=model, loss_func=loss_func, opt=optimizer)

In [14]:
torch.save(model.state_dict(), './saved_model')

In [ ]:
#-------------------------------------- 

In [ ]:
from time import time
print("check")
train_loader = torch.utils.data.DataLoader(my_dataset,shuffle=True,num_workers=2,batch_size=64,pin_memory=True)
start = time()

for epoch in range(1, 3):
    for x,y in train_loader:
        x = x.to(device='cuda')
        y = y.to(device='cuda')
end = time()
print("{} s".format(end - start))]
mp.cpu_count()
# np_data = np.load('../../Data/PSSM_train_data.npz')
# for i in range(10):
embeddings = torch.load('../../Data/training_embeddings/'+np_data.files[2]+'.pt').transpose(0,1)
# labels = np_data[np_data.files[2]][2]
# labels = torch.from_numpy(labels)

embeddings = torch.nn.functional.pad(embeddings, (700 - embeddings.shape[1], 0), 'constant', 0)
labels     = torch.nn.functional.pad(labels, (700 - labels.shape[0], 1), 'constant', -9)
# print(embeddings.shape)
# print(labels.shape)
# import os
# sum([os.path.getsize('../../Data/training_embeddings/'+f)/1000000 for f in os.listdir('../../Data/training_embeddings/')])
# files = os.listdir('../../Data/training_embeddings/')
# [os.path.isfile('../../Data/training_embeddings/'+f) for f in files]
# os.path.getsize('../../Data/training_embeddings/1G5A_1_A.pt')
# load_file = torch.load('../../Data/training_embeddings/'+np_data.files[2]+'.pt').transpose(0,1)
# load_file.shape
new_lab = torch.full(700, -9)
new_lab[]
# embeddings = embeddings.transpose(0,1)
# embeddings = torch.cat ((torch.full((1,1280),0), embeddings, torch.full((1,1280),0)))
# print(embeddings[:, 0].size())
# print(embeddings[0,:])
# print(torch.load('../../Data/training_embeddings/'+np_data.files[2]+'.pt').transpose(0,1)[0:,])
# out = torch.nn.functional.pad(embeddings, (700 - 1 - embeddings.shape[1],1), 'constant', 0)
# np_data = np.load('../../Data/PSSM_train_data.npz')
# for i in range(10):
embeddings = torch.load('../../Data/training_embeddings/'+np_data.files[2]+'.pt').transpose(0,1)
# labels = np_data[np_data.files[2]][2]
# labels = torch.from_numpy(labels)

embeddings = torch.nn.functional.pad(embeddings, (700 - embeddings.shape[1], 0), 'constant', 0)
labels     = torch.nn.functional.pad(labels, (700 - labels.shape[0], 1), 'constant', -9)
# print(embeddings.shape)
# print(labels.shape)
# import os
# sum([os.path.getsize('../../Data/training_embeddings/'+f)/1000000 for f in os.listdir('../../Data/training_embeddings/')])
# files = os.listdir('../../Data/training_embeddings/')
# [os.path.isfile('../../Data/training_embeddings/'+f) for f in files]
# os.path.getsize('../../Data/training_embeddings/1G5A_1_A.pt')

np_data = np.load('../../Data/PSSM_train_data.npz')
my_dataset = np_dataset(np_data)
train_loader = torch.utils.data.DataLoader(my_dataset,shuffle=True,num_workers=2,batch_size=64,pin_memory=True)
# label = torch.from_numpy(np_data[np_data.files[1]][2])
# torch.cuda.is_available()
# np_data = np.load('../../Data/PSSM_train_data.npz')
# test_sample = np_data[np_data.files[0]]
# PSSM = torch.from_numpy(test_sample[3:])
# PSSM.shape

# label = torch.nn.functional.pad(label, (700- 1 - label.shape[-1], 1))
# print(label.shape)
# print(PSSM.shape)
# x = torch.nn.functional.pad(PSSM, (700 - 1 - PSSM.shape[1],1))
# print(x.shape)

# PSSM = torch.from_numpy(np_data[np_data.files[1]][3:])
# PSSM.shape[-1]

# x = torch.zeros((21,720))
# print(x.shape)
# x=x[:,:700]
# print(x.shape)

['2ZTG_1_A', '5IES_3_C', '3SUZ_1_A', '1MZ9_1_A', '2N8O_1_A']

In [ ]:
class CNN_BRNN(nn.Module):
    def __init__(self, n_in=1280, n_state=3, conv_hidden_size=64, rnn_hidden_size=256):
        super().__init__()

        # Conv1d(in_channels, out_channels, kernel_size, stride, padding)
        self.conv1 = nn.Sequential(nn.Conv1d( n_in, conv_hidden_size, 3, 1, 3 // 2), nn.ReLU())
        self.conv2 = nn.Sequential(nn.Conv1d( n_in, conv_hidden_size, 7, 1, 7 // 2), nn.ReLU())
        self.conv3 = nn.Sequential(nn.Conv1d( n_in, conv_hidden_size, 11, 1, 11 // 2), nn.ReLU())

        # nn.GRU(n_input, n_hidden, n_layers, bias, batch_first, dropout, bidirectional)
        # bias – If False, then the layer does not use bias weights b_ih and b_hh. Default: True
        # batch_first – If True, then the input and output tensors are provided as (batch, seq, feature) instead of (seq, batch, feature).
        #               Note that this does not apply to hidden or cell states.
        # dropout – If non-zero, introduces a Dropout layer on the outputs of each GRU layer except the last layer, with dropout probability equal to dropout. Default: 0
        # bidirectional – If True, becomes a bidirectional GRU. Default: False

        self.brnn = nn.GRU(conv_hidden_size*3, rnn_hidden_size, 3, True, True, 0.5, True)

        self.fc = nn.Sequential(
                nn.Linear(rnn_hidden_size*2+conv_hidden_size*3, 128),
                nn.ReLU(),
                nn.Linear(128, n_state))

    def forward(self, x):
        # (batch_size , hidden_size , seq_len)
        conv_out = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x)], dim=1)
        # (batch_size , seq_len , hidden_size)
        conv_out = conv_out.transpose(1, 2)

        # bidirectional rnn
        out, _ = self.brnn(conv_out)

        out = torch.cat([conv_out, out], dim=2)
        # print(out.sum())

        out = self.fc(out)
        #  (N x L x C) --> (N x C x L)
        out = out.transpose(1,2)
        return out


In [ ]:
#----- training loop
def training_loop(epochs, np_data, model, loss_func, opt):

  # iterations = len(train_dataset)//batch_size #len(dataset)//batch_size
  model.train()

  for epoch in range(epochs):
    print(f"Epoch[{epoch+1}/{epochs}]\n")


    acc_num = 0
    acc_den = 0
    batch = 0

    #load half the training data to be used for each epoch into RAM

    if epoch%2 == 0 :       # randomly pick 1/2 the files to use for the epoch. Retain the other 1/2 for next epoch
      rand_index = np.arange(21005)
      np.random.shuffle(rand_index)
      include_chains = rand_index[:10503]
    else:
      include_chains = rand_index[10503:]

    print("loading data {}".format(len(include_chains)))
    RAM_tensor_dict = {np_data.files[x] : torch.load('/home/training_embeddings/training_embeddings/'+np_data.files[x]+'.pt') for x in include_chains}#use the other half
    print("data loaded")
    my_dataset = RAM_dataset(RAM_tensor_dict, np_data)
    batch_size=64
    dataloader = torch.utils.data.DataLoader(my_dataset,shuffle=True,batch_size=batch_size) #torch.save zips the files, and pytorch zip module has some problem when using multiple workers (when two batches try to open the same filename)

    for x,y in dataloader:      
      # batch += 1
      # print("batch: {}".format(batch))
      #forward
      x= x.to(device='cuda')
      y= y.to(device='cuda')
      outputs    = model(x)

      # print(outputs.shape)
      # output : (N x C x L)
      _, preds   = torch.max(outputs, 1)

      # print(preds.shape)
      # print(y.shape)
      acc_num += (preds == y).sum().item() 
      acc_den += len(y[y > -1])

      # y = y.view(-1)
      loss = loss_func(outputs, y)
      # cum_loss += loss.item()

      #backward
      opt.zero_grad() 
      loss.backward()

      #adam step
      opt.step()

      #scheduler step

    print(f'Accuracy: {(acc_num/acc_den):.3f}\n')
    del RAM_tensor_dict
    del my_dataset
    del dataloader
    gc.collect()
    # for x,y in validation_loader:
